# Model Training — Advisor Risk Scoring
Reads the pre-built training table from `build_features`, runs Optuna hyperparameter tuning
on a LightGBM classifier (median imputation), refits the best params, and registers the best
model to Unity Catalog. All trials are logged to MLflow as nested runs.

**Alias (`model_alias` widget / job parameter)**
* Default is `challenger` — manual and automated retraining both register a challenger.
* Bootstrap: if the model has no `@champion` yet, the new version gets **both** `@champion` and `@challenger`.
* Promotion to `@champion` happens only in the evaluation / A/B step, never silently in training.

In [0]:
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from lightgbm import LGBMClassifier
from src.config import CFG
from src.utils import get_logger

logger = get_logger("models.train")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CFG.experiment_name)

# ── Alias to assign to the newly trained model (job parameter overrides widget) ──
dbutils.widgets.text("model_alias", CFG.model_challenger_alias)
MODEL_ALIAS = dbutils.widgets.get("model_alias").strip().lower()

ALLOWED_ALIASES = {CFG.model_champion_alias, CFG.model_challenger_alias}
if MODEL_ALIAS not in ALLOWED_ALIASES:
    raise ValueError(f"model_alias must be one of {sorted(ALLOWED_ALIASES)}, got '{MODEL_ALIAS}'")
logger.info(f"New model will be registered as @{MODEL_ALIAS}")

In [0]:
LABEL_COL = "at_risk"
ID_COL = "Salesforce_Id__c"

FEATURE_COLS = [
    # Book profile — benchmark ratios
    "avg_advisor_value", "avg_vs_team_ratio", "avg_vs_region_ratio",
    "metrics_below_team", "metrics_below_region",
    "pct_below_team", "pct_below_region",
    # Tool adoption
    "total_advisor_sessions", "tools_adopted_count", "avg_sessions_vs_team",
    # Holdings — portfolio concentration
    "total_aum", "num_funds", "max_fund_pct", "hhi_concentration",
]

In [0]:
# ── Read the pre-built training table (from build_features notebook) ──
# Same feature logic as the serving lookup table, so no training/serving skew.
pdf = spark.table(CFG.fq("training")).toPandas()
logger.info(f"Loaded {len(pdf):,} rows from {CFG.fq('training')}  |  at_risk rate: {pdf[LABEL_COL].mean():.4f}")

In [0]:
X = pdf[FEATURE_COLS]
y = pdf[LABEL_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
logger.info(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  at_risk rate: {y.mean():.4f}")
logger.info(f"Features: {len(FEATURE_COLS)}")

In [0]:
def build_pipeline(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", LGBMClassifier(**params, random_state=42, verbose=-1)),
    ])


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    pipe = build_pipeline(params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        mlflow.log_params(params)
        mlflow.log_metric("cv_recall_mean", scores.mean())
        mlflow.log_metric("cv_recall_std", scores.std())

    return scores.mean()


with mlflow.start_run(run_name="lgbm_optuna_tuning") as parent_run:
    study = optuna.create_study(direction="maximize", study_name="advisor_risk_lgbm")
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    # Refit best pipeline on full training set
    best_params = study.best_params
    best_pipeline = build_pipeline(best_params)
    best_pipeline.fit(X_train, y_train)

    # Evaluate on held-out test set
    y_pred = best_pipeline.predict(X_test)
    y_prob = best_pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_prob),
        "best_cv_recall": study.best_value,
    }

    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
    mlflow.log_metrics(metrics)

    mlflow.set_tag("model_alias_requested", MODEL_ALIAS)

    # Log + register the refit best pipeline in the parent run (lineage: version -> tuning study)
    signature = infer_signature(X_test, y_pred)
    model_info = mlflow.sklearn.log_model(
        best_pipeline,
        name="model",
        registered_model_name=CFG.fq_model_name,
        input_example=X_test.head(5),
        signature=signature,
    )

    logger.info(f"Best trial: #{study.best_trial.number} | CV Recall: {study.best_value:.4f}")
    logger.info(f"Best params: {best_params}")
    logger.info(f"Test metrics: {metrics}")
    logger.info(f"Parent Run ID: {parent_run.info.run_id}")
    logger.info(f"Registered {CFG.fq_model_name} version {model_info.registered_model_version}")

In [0]:
logger.info(f"\n{classification_report(y_test, y_pred, target_names=['healthy', 'at_risk'], zero_division=0)}")

# Top 10 trials
trials_df = study.trials_dataframe()[
    ["number", "value", "params_n_estimators", "params_max_depth",
     "params_learning_rate", "params_num_leaves"]
]
trials_df = (
    trials_df.sort_values("value", ascending=False)
    .head(10)
    .rename(columns={"value": "cv_recall"})
)
display(trials_df)

In [0]:
client = MlflowClient()
new_version = model_info.registered_model_version


def alias_exists(model_name: str, alias: str) -> bool:
    """True if the alias is currently set on any version of the model."""
    try:
        client.get_model_version_by_alias(model_name, alias)
        return True
    except Exception:
        return False


# Bootstrap: first ever model (or new environment) -> make it both champion and challenger,
# so downstream pyfunc / deploy / A/B steps have both aliases to resolve.
if not alias_exists(CFG.fq_model_name, CFG.model_champion_alias):
    aliases_to_set = [CFG.model_champion_alias, CFG.model_challenger_alias]
    logger.info(f"No @{CFG.model_champion_alias} found — bootstrapping version {new_version} as both aliases")
else:
    aliases_to_set = [MODEL_ALIAS]

for alias in aliases_to_set:
    client.set_registered_model_alias(CFG.fq_model_name, alias, new_version)
    logger.info(f"{CFG.fq_model_name} version {new_version} aliased as @{alias}")

logger.info(f"Best params: {best_params}")